# Quadragon - PPO Training (Colab side)

**Workflow:** Colab does the GPU training; **evaluation (video, gait analysis)
runs locally** because Colab's headless renderer errors on video export. This
notebook trains + checkpoints to Drive, then you sync the Drive folder down to
your local machine and run `analyze_checkpoint.py` / rollout video there.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is fine).

Drive project folder should contain the env files for whatever version you're
training, plus shared files:
`quadragon_env.py`, `quadragon_env_v2.py`, `quadragon_env_v3a.py`,
`quadragon_env_v3b.py`, `train.py`, `smoke_test.py`, `compare_checkpoints.py`,
`quadragon_mg995.xml`, and the 13 STLs next to the xml.

Set `VERSION` in the next section. Everything downstream reads it.

**Checkpoints save directly to Drive** - Colab kills sessions without warning,
so local-disk checkpoints can vanish. The symlink in the training cell handles this.

## 1. Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT if your Drive folder lives elsewhere
DRIVE_PROJECT = '/content/drive/MyDrive/quadragon_v1'

# EDIT: which version to train
#   'v1'  - minimal baseline
#   'v2'  - + foot clearance
#   'v3a' - + data-driven coordination (duty balance + freq penalty)
#   'v3b' - + phase-clock coordination (prescribed diagonal trot)
#   'v4'  - v2 + random push perturbations (dynamic-balance pressure)
#   'v5'  - v4 + realistic MG995 servo dynamics (kills vibration gait)
VERSION = 'v5'
RUN_NAME = f'{VERSION}_colab'

import os
assert os.path.exists(f'{DRIVE_PROJECT}/quadragon_mg995.xml'), f'Model xml not found in {DRIVE_PROJECT}'

# Version-specific env file must be present
_needed = {
    'v1':  'quadragon_env.py',
    'v2':  'quadragon_env_v2.py',
    'v3a': 'quadragon_env_v3a.py',
    'v3b': 'quadragon_env_v3b.py',
    'v4':  'quadragon_env_v4.py',
    'v5':  'quadragon_env_v5.py',
}[VERSION]
assert os.path.exists(f'{DRIVE_PROJECT}/{_needed}'), f'{_needed} not found in Drive folder - upload it first'
print('Drive folder found:', os.listdir(DRIVE_PROJECT))
print(f'Training VERSION={VERSION}, RUN_NAME={RUN_NAME}')

## 2. Copy project to local disk
Training reads the model xml + meshes constantly — running from Drive directly is slow and flaky. Code and meshes go local; only checkpoints go back to Drive.

In [ ]:
!rm -rf /content/quadragon_work
!cp -r "$DRIVE_PROJECT" /content/quadragon_work
%cd /content/quadragon_work
!ls

## 3. Install dependencies

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 tensorboard

# Headless rendering backend for rollout videos later (Colab GPU runtimes support EGL)
import os
os.environ['MUJOCO_GL'] = 'egl'

import mujoco, torch
print('MuJoCo:', mujoco.__version__)
print('CUDA available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 4. Smoke test
Same five checks that passed in development — run once per fresh session to confirm the transfer to Colab is clean before burning GPU time.

In [ ]:
import subprocess
subprocess.run(['python3', 'smoke_test.py', '--version', VERSION], check=True)

## 5. TensorBoard (start before training so it's live during the run)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Train
Checkpoints write straight to Drive every ~100k steps via symlink, so a dead session costs you at most the steps since the last checkpoint.

PPO itself is a small MLP — the env stepping (CPU) is the bottleneck, not the GPU, so `n_envs` matters more than GPU model here. 8 is right for Colab's 2-core CPU; on the 4090 machine (more cores) push it to 16-32.

In [ ]:
# Checkpoints -> Drive (survives disconnects)
DRIVE_CKPT = f'{DRIVE_PROJECT}/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
!rm -rf checkpoints && ln -s "$DRIVE_CKPT" checkpoints

!python3 train.py --version $VERSION --n-envs 8 --timesteps 3000000 --run-name $RUN_NAME

## 7. Save final model + normalization stats to Drive
Both files. The policy is meaningless without its matching VecNormalize stats.

In [ ]:
!cp quadragon_${RUN_NAME}_final.zip vecnormalize_${RUN_NAME}.pkl "$DRIVE_PROJECT/" 2>/dev/null && echo 'Saved to Drive' || echo 'Final files not found - if the session died mid-run, grab the latest from checkpoints/ instead'

## 7. Training done - evaluation happens LOCALLY

Colab's headless renderer errors on video export, so gait analysis and rollout
video run on your local machine instead. After training finishes:

1. **Sync the Drive folder down to local** (Drive desktop client, or re-download
   the `checkpoints/` folder + the final `.zip`/`.pkl`).
2. **Locally**, find the best checkpoint (not the final one):
   ```bash
   python3 compare_checkpoints.py    # set RUN inside it to match this run
   ```
3. **Locally**, analyze the best checkpoint - this is the real comparison:
   ```bash
   python3 analyze_checkpoint.py <best>.zip <best_vecnorm>.pkl --version {VERSION} --save-plot {VERSION}_analysis.png
   ```
   (use the same VERSION you trained: v3a analyzes with --version v3a, etc.)

The cell below just confirms the final files exist in Drive before you sync.

In [ ]:
# Confirm final model + vecnorm are in Drive, ready to sync locally.
# (Checkpoints are already in Drive via the symlink; this checks the final pair.)
import os, glob

final_zip = f'quadragon_{RUN_NAME}_final.zip'
final_pkl = f'vecnormalize_{RUN_NAME}.pkl'

for f in (final_zip, final_pkl):
    if os.path.exists(f):
        import shutil
        shutil.copy(f, DRIVE_PROJECT)
        print(f'Copied {f} -> Drive')
    else:
        print(f'{f} not found (session may have died mid-run - use latest checkpoints/ instead)')

# Show what checkpoints exist in Drive for this run, so you know what to sync
ckpts = sorted(glob.glob(f'{DRIVE_PROJECT}/checkpoints/quadragon_{RUN_NAME}_*_steps.zip'))
print(f'\n{len(ckpts)} checkpoints in Drive for {RUN_NAME}:')
for c in ckpts[-5:]:
    print(' ', os.path.basename(c))
if len(ckpts) > 5:
    print(f'  ... and {len(ckpts)-5} earlier')

## Resuming after a disconnect
Re-run cells 1-5, then instead of a fresh `train.py`, load the latest checkpoint:
```python
import glob
latest = sorted(glob.glob(f'checkpoints/quadragon_{RUN_NAME}_*_steps.zip'))[-1]
print('Resuming from', latest)
```
then: `model = PPO.load(latest, env=venv)` + `model.learn(remaining, reset_num_timesteps=False)`
- or just restart if you were early in training (before ~500k, cheap to redo).

`target_kl=0.02` is baked into `train.py` for all versions, so the v1-style
clip_fraction runaway shouldn't recur. If it somehow does, `resume_with_fix.py`
from the v1 debugging session applies the same technique to any version.